In [0]:
from pyspark.sql.functions import coalesce, col, lit

In [0]:
in_catalog = dbutils.widgets.get("in_catalog")
in_schema = dbutils.widgets.get("in_schema")
out_catalog = dbutils.widgets.get("out_catalog")
out_schema = dbutils.widgets.get("out_schema")

In [0]:
products = spark.read.table(f"{in_catalog}.{in_schema}.tbl_olist_products_dataset")
translations = spark.read.table(f"{in_catalog}.{in_schema}.tbl_product_category_name_translation")

In [0]:
products_join = products.join(translations, on="product_category_name", how="left")

## Calculated columns

In [0]:
products_calc = (
    products_join
    .withColumn("product_volume_cm3",
        col("product_length_cm").cast("int")
        * col("product_height_cm").cast("int")
        * col("product_width_cm").cast("int"))
    .withColumn("density_g_cm3", col("product_weight_g").cast("int") / col("product_volume_cm3"))
)

In [0]:
products_transformed = products_calc.select(
    "product_id",
    coalesce(col("product_category_name"), lit("desconhecido")).alias("product_category_name"),
    coalesce(col("product_category_name_english"), lit("unknown")).alias("product_category_name_english"),
    col("product_name_lenght").cast("int").alias("product_name_length"),
    col("product_description_lenght").cast("int").alias("product_description_length"),
    col("product_photos_qty").cast("int"),
    col("product_weight_g").cast("int"),
    "product_volume_cm3",
    "density_g_cm3",
)

In [0]:
(
    products_transformed
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{out_catalog}.{out_schema}.products")
)